# Benchmark 1: 2 Class vs 4 Class: Cross-Session

In [1]:
import numpy as np
from moabb.datasets import BNCI2014_001
from moabb.paradigms import MotorImagery, LeftRightImagery
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from mne.decoding import CSP
from moabb.evaluations import CrossSubjectEvaluation
from sklearn.pipeline import make_pipeline
from scipy import signal
from scipy.io import loadmat
import os
import mne
from scipy.linalg import logm, expm
from sklearn.svm import SVC
from sklearn.metrics import balanced_accuracy_score
from pyriemann.estimation import Covariances
import scipy
from pyriemann.tangentspace import TangentSpace

In [2]:
from pyriemann.utils import mean_riemann
from scipy.optimize import minimize
from pymanopt import Problem
from pymanopt.manifolds import SpecialOrthogonalGroup
from pymanopt.optimizers import SteepestDescent
from pymanopt import Problem
from functools import partial
from pymanopt.function import numpy as pymanopt_numpy
import autograd.numpy as anp  # Autograd's NumPy replacement
from autograd import grad
import pymanopt
import autograd.scipy.linalg as linalg
from pyriemann.classification import MDM
import numpy as np
from pyriemann.estimation import Covariances
from pyriemann.utils.mean import mean_riemann
from scipy.linalg import fractional_matrix_power, logm, eigh
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import accuracy_score
from sklearn.neighbors import NearestNeighbors

In [3]:
def encode_labels(labels_list):
    encoded_list = []
    for labels in labels_list:
        # Create mapping from original labels to 0,1
        unique_labels = np.unique(labels)
        label_map = {unique_labels[i]: i for i in range(len(unique_labels))}
        
        # Apply mapping
        encoded = np.array([label_map[label] for label in labels])
        encoded_list.append(encoded)
    return encoded_list

In [4]:
# Define a causal bandpass filter function using a Butterworth design.
def causal_bandpass_filter(data, lowcut=8, highcut=30, fs=250, order=5):
    nyq = 0.5 * fs
    # Normalize the cutoff frequencies (Matlab's fir1 expects normalized cutoff frequencies
    low = lowcut / nyq
    high = highcut / nyq
    # Design the FIR filter. Note: order+1 coefficients are returned to match Matlab's fir1 which returns n+1 taps.
    b = signal.firwin(order + 1, [low, high], window='hamming', pass_zero=False)
    # Apply the filter causally using lfilter (this introduces a constant delay).
    filtered_data = signal.lfilter(b, [1.0], data)
    return filtered_data

In [5]:
# With a sampling frequency of 250 Hz, 1001 samples equate to 1001/250 seconds.
sfreq = 250
tmin = 0.5       # Epoch start at cue onset.
# Set tmax so that n_samples = (tmax-tmin)*sfreq + 1 = 1001, i.e. 4 seconds long.
tmax = 3.5  # This gives 4.0 seconds.
filter_order = 50

In [9]:
def twofour_crosssession(n_classes):

    data_dir = '/home/vishwa/eeg_tl/Recreating papers/BCI2b/BCICIV_2b_gdf'

    # Lists to hold data for all subjects
    train_active_X = []         # List to hold numpy arrays with shape (n_trials, n_channels, n_times) per subject
    train_active_y = []         # List to hold event labels per subject
    train_active_metadata = []  # List to hold event metadata per subject

    # Define subject IDs (B01 to B09)
    subjects = [f'B{subj:02d}' for subj in range(1, 10)]

    for subj in subjects:
        # Define training sessions for this subject (e.g., B0101T.gdf, B0102T.gdf, B0103T.gdf)
        session_ids = ['01T'] #, '02T', '03T']
        subj_epochs_list = []

        for sess in session_ids:
            filename = os.path.join(data_dir, f'{subj}{sess}.gdf')
            
            # Check if file exists to avoid errors
            if not os.path.exists(filename):
                print(f"File {filename} not found, skipping.")
                continue
            
            # Load the GDF file
            raw = mne.io.read_raw_gdf(filename, preload=True, verbose=False)
            
            # Extract events from annotations, mapping '769' to 1 (left) and '770' to 2 (right)
            event_id_mapping = {'769': 1, '770': 2}
            events, event_dict = mne.events_from_annotations(raw, event_id=event_id_mapping, verbose=False)
            
            # Select only EEG channels (C3, Cz, C4)
            print(raw.ch_names)
            eeg_channels = [ch for ch in raw.ch_names if ch in ['EEG:C3', 'EEG:Cz', 'EEG:C4']]
            if len(eeg_channels) != 3:
                print(f"Warning: Expected 3 EEG channels for {subj}{sess}, found {len(eeg_channels)}: {eeg_channels}")
            raw_eeg = raw.pick_channels(eeg_channels, verbose=False)
            
            # Define epoching parameters (consistent with Dataset 2a)
            tmin = 0.5  # seconds after cue onset
            tmax = 3.5  # seconds after cue onset
            
            # Create epochs
            epochs = mne.Epochs(
                raw_eeg,
                events,
                event_id={'left': 1, 'right': 2},
                tmin=tmin,
                tmax=tmax,
                baseline=None,  # No baseline correction, matching your 2a code
                preload=True,
                verbose=False
            )
            
            subj_epochs_list.append(epochs)
        
        # Skip subject if no sessions were processed
        if not subj_epochs_list:
            print(f"No valid sessions found for subject {subj}, skipping.")
            continue
        
        # Concatenate epochs across sessions for this subject
        if len(subj_epochs_list) > 1:
            subj_epochs = mne.concatenate_epochs(subj_epochs_list, verbose=False)
        else:
            subj_epochs = subj_epochs_list[0]
        
        # Get the epoch data
        subj_data = subj_epochs.get_data()
        
        # Apply causal bandpass filter to each trial and channel (matching Dataset 2a)
        n_trials, n_channels, n_times = subj_data.shape
        fs = raw.info['sfreq']  # Sampling frequency (250 Hz for Dataset 2b)
        subj_filtered_data = np.empty_like(subj_data)
        for trial in range(n_trials):
            for ch in range(n_channels):
                subj_filtered_data[trial, ch, :] = causal_bandpass_filter(
                    subj_data[trial, ch, :],
                    lowcut=8,   # Lower bound of sensorimotor rhythm
                    highcut=30, # Upper bound of sensorimotor rhythm
                    fs=fs,
                    order=50    # Filter order
                )
        
        # Append processed data, labels, and metadata
        train_active_X.append(subj_filtered_data[:120])
        train_active_y.append(subj_epochs.events[:120, 2])  # Labels in third column (1 or 2)
        train_active_metadata.append(subj_epochs.events)
        
        # Print shape to verify
        print(f"Subject {subj}: Epoch data shape {subj_filtered_data.shape}")

    print(f"Loaded data for {len(train_active_X)} subjects.")


    # Lists to hold data for all subjects
    eval_active_X = []         # List to hold numpy arrays with shape (n_trials, n_channels, n_times) per subject
    eval_active_y = []         # List to hold event labels per subject
    eval_active_metadata = []  # List to hold event metadata per subject

    # Define subject IDs (B01 to B09)
    subjects = [f'B{subj:02d}' for subj in range(1, 10)]

    for subj in subjects:
        # Define training sessions for this subject (e.g., B0101T.gdf, B0102T.gdf, B0103T.gdf)
        session_ids = ['02T'] #, '02T', '03T']
        subj_epochs_list = []

        for sess in session_ids:
            filename = os.path.join(data_dir, f'{subj}{sess}.gdf')
            
            # Check if file exists to avoid errors
            if not os.path.exists(filename):
                print(f"File {filename} not found, skipping.")
                continue
            
            # Load the GDF file
            raw = mne.io.read_raw_gdf(filename, preload=True, verbose=False)
            
            # Extract events from annotations, mapping '769' to 1 (left) and '770' to 2 (right)
            event_id_mapping = {'769': 1, '770': 2}
            events, event_dict = mne.events_from_annotations(raw, event_id=event_id_mapping, verbose=False)
            
            # Select only EEG channels (C3, Cz, C4)
            print(raw.ch_names)
            eeg_channels = [ch for ch in raw.ch_names if ch in ['EEG:C3', 'EEG:Cz', 'EEG:C4']]
            if len(eeg_channels) != 3:
                print(f"Warning: Expected 3 EEG channels for {subj}{sess}, found {len(eeg_channels)}: {eeg_channels}")
            raw_eeg = raw.pick_channels(eeg_channels, verbose=False)
            
            # Define epoching parameters (consistent with Dataset 2a)
            tmin = 0.5  # seconds after cue onset
            tmax = 3.5  # seconds after cue onset
            
            # Create epochs
            epochs = mne.Epochs(
                raw_eeg,
                events,
                event_id={'left': 1, 'right': 2},
                tmin=tmin,
                tmax=tmax,
                baseline=None,  # No baseline correction, matching your 2a code
                preload=True,
                verbose=False
            )
            
            subj_epochs_list.append(epochs)
        
        # Skip subject if no sessions were processed
        if not subj_epochs_list:
            print(f"No valid sessions found for subject {subj}, skipping.")
            continue
        
        # Concatenate epochs across sessions for this subject
        if len(subj_epochs_list) > 1:
            subj_epochs = mne.concatenate_epochs(subj_epochs_list, verbose=False)
        else:
            subj_epochs = subj_epochs_list[0]
        
        # Get the epoch data
        subj_data = subj_epochs.get_data()
        
        # Apply causal bandpass filter to each trial and channel (matching Dataset 2a)
        n_trials, n_channels, n_times = subj_data.shape
        fs = raw.info['sfreq']  # Sampling frequency (250 Hz for Dataset 2b)
        subj_filtered_data = np.empty_like(subj_data)
        for trial in range(n_trials):
            for ch in range(n_channels):
                subj_filtered_data[trial, ch, :] = causal_bandpass_filter(
                    subj_data[trial, ch, :],
                    lowcut=8,   # Lower bound of sensorimotor rhythm
                    highcut=30, # Upper bound of sensorimotor rhythm
                    fs=fs,
                    order=50    # Filter order
                )
        
        # Append processed data, labels, and metadata
        eval_active_X.append(subj_filtered_data[:120])
        eval_active_y.append(subj_epochs.events[:120, 2])  # Labels in third column (1 or 2)
        eval_active_metadata.append(subj_epochs.events)
        
        # Print shape to verify
        print(f"Subject {subj}: Epoch data shape {subj_filtered_data.shape}")

    print(f"Loaded data for {len(train_active_X)} subjects.")
    
    train_active_y = encode_labels(train_active_y)
    eval_active_y = encode_labels(eval_active_y)

        # Initialize list to hold tangent space features for each subject
    train_aligned = []

    # Loop over all 9 subjects
    for subj in range(9):
        # Compute covariance matrices for each trial (shape: 144, 22, 22)
        cov_est = Covariances(estimator='scm')
        P = cov_est.fit_transform(train_active_X[subj])  # Input: (144, 22, 501), Output: (144, 22, 22)

        ts = TangentSpace(metric='riemann')
        X_ts = ts.fit_transform(P)  # Returned shape: (n_trials, feature_dim); for 22 channels, feature_dim is 253
        
        train_aligned.append(X_ts.T)

    eval_aligned = []

    # Loop over all 9 subjects
    for subj in range(9):
        # Compute covariance matrices for each trial (shape: 144, 22, 22)
        cov_est = Covariances(estimator='scm')
        P = cov_est.fit_transform(train_active_X[subj])  # Input: (144, 22, 501), Output: (144, 22, 22)

        ts = TangentSpace(metric='riemann')
        X_ts = ts.fit_transform(P)  # Returned shape: (n_trials, feature_dim); for 22 channels, feature_dim is 253
        
        eval_aligned.append(X_ts.T)

    align_per_class = 12
    n_subjects = 9
    if(n_classes==2):
        classes = [0, 1]
    else:
        classes = [0, 1, 2, 3]  # Two classes as per your setup
    epsilon = 1e-6
    n_channels = 3
    N = 5
    alpha = 0.01
    beta = 0.1
    rho = 20
    p = 10
    d = 6
    accuracies = []

    for subj_idx in range(len(train_active_X)):
        # print(subj_idx)
        # Split data into train/test using leave-one-subject-out
        X_target = train_aligned[subj_idx]
        y_target = train_active_y[subj_idx]
        
        # Concatenate data from other subjects
        source_subjects = [j for j in range(9) if j != subj_idx]
        X_source = np.hstack([train_aligned[j] for j in source_subjects])  # Shape: (253, 1152)
        y_source = np.concatenate([train_active_y[j] for j in source_subjects])

        n_s = X_source.shape[1]  # 1152
        n_t = X_target.shape[1]  # 144

        # Initial pseudo-labels
        clf_init = LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto')
        clf_init.fit(X_source.T, y_source)
        hat_y_t = np.zeros_like(clf_init.predict(X_target.T))

        # Iterative optimization
        for n in range(N):
            # Source domain scatter matrices
            m_0 = X_source[:, y_source == 0].mean(axis=1)  # Shape: (253,)
            m_1 = X_source[:, y_source == 1].mean(axis=1)
            if (n_classes==4):  # Shape: (253,)
                m_2 = X_source[:, y_source == 2].mean(axis=1)  # Shape: (253,)
                m_3 = X_source[:, y_source == 3].mean(axis=1)
            m = X_source.mean(axis=1)  # Shape: (253,)
            n_0 = np.sum(y_source == 0)
            n_1 = np.sum(y_source == 1)
            if(n_classes==4):
                n_2 = np.sum(y_source == 2)
                n_3 = np.sum(y_source == 3)
            S_b = n_0 * np.outer(m_0 - m, m_0 - m) + n_1 * np.outer(m_1 - m, m_1 - m)  # Shape: (253, 253)
            if (n_classes==4):
                S_b = n_0 * np.outer(m_0 - m, m_0 - m) + n_1 * np.outer(m_1 - m, m_1 - m) + n_2 * np.outer(m_2 - m, m_2 - m) + n_3 * np.outer(m_3 - m, m_3 - m)
            S_w = np.zeros((d, d))  # Shape: (253, 253)
            for k_class in classes:
                X_k = X_source[:, y_source == k_class]  # Shape: (253, n_k)
                S_w_k = np.cov(X_k, rowvar=True) * (X_k.shape[1] - 1)  # Shape: (253, 253)
                S_w += S_w_k

            # Target domain similarity matrix
            nn = NearestNeighbors(n_neighbors=10)
            nn.fit(X_target.T)
            distances, indices_nn = nn.kneighbors(X_target.T)
            sigma = 1.0
            S = np.zeros((n_t, n_t))  # Shape: (144, 144)
            for i in range(n_t):
                for j in indices_nn[i]:
                    if j != i:
                        S[i, j] = np.exp(-np.sum((X_target[:, i] - X_target[:, j])**2) / (2 * sigma**2))
                        S[j, i] = S[i, j]

            # Laplacian matrix
            D = np.diag(np.sum(S, axis=1))
            D_inv_sqrt = np.diag(1.0 / np.sqrt(np.diag(D)))
            # L_target = np.eye(n_t) - D_inv_sqrt @ S @ D_inv_sqrt
            L_target = D - S
            # Shape: (144, 144)

            # Centering matrix
            H = np.eye(n_t) - np.ones((n_t, n_t)) / n_t  # Shape: (144, 144)

            # One-hot encodings
            Y_s = np.eye(n_classes)[y_source]  # Shape: (1152, 2)
            hat_Y_t = np.eye(n_classes)[hat_y_t]  # Shape: (144, 2)
            N_s = Y_s / n_s  # Shape: (1152, 2)
            # N_t = hat_Y_t / n_t  # Shape: (144, 2)
            N_t = np.zeros((n_t, n_classes)) if n == 0 else np.eye(n_classes)[hat_y_t] / n_t

            # Joint probability MMD matrix R
            R11 = X_source @ N_s @ N_s.T @ X_source.T  # Shape: (253, 253)
            R12 = -X_source @ N_s @ N_t.T @ X_target.T  # Shape: (253, 253)
            R21 = -X_target @ N_t @ N_s.T @ X_source.T  # Shape: (253, 253)
            R22 = X_target @ N_t @ N_t.T @ X_target.T  # Shape: (253, 253)
            R = np.block([[R11, R12], [R21, R22]])  # Shape: (506, 506)

            # Other matrices
            P = np.block([[S_w, np.zeros((d, d))], [np.zeros((d, d)), np.zeros((d, d))]])  # Shape: (506, 506)
            L = np.block([[np.zeros((d, d)), np.zeros((d, d))], [np.zeros((d, d)), X_target @ L_target @ X_target.T]])  # Shape: (506, 506)
            I_d = np.eye(d)
            U = np.block([[I_d, -I_d], [-I_d, 2 * I_d]])  # Shape: (506, 506)
            V = np.block([[np.zeros((d,d)), np.zeros((d,d))], [np.zeros((d,d)), X_target @ H @ X_target.T]]) # Shape: (506, 506) - 

            # Optimization
            A = alpha * P + beta * L + rho * U + R  # Shape: (506, 506)
            B = V + 1e-6 * np.eye(2 * d)  # Shape: (506, 506)
            eigvals, eigvecs = eigh(A, B)
            W = eigvecs[:, :p]  # Shape: (506, 10)
            A_proj = W[:d, :]  # Shape: (253, 10)
            B_proj = W[d:, :]  # Shape: (253, 10)

            # Train and predict
        
            clf = LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto')
            clf.fit((A_proj.T @ X_source).T, y_source)
            hat_y_t = clf.predict((B_proj.T @ X_target).T)

        # Compute accuracy
        acc = balanced_accuracy_score(y_target, hat_y_t)
        accuracies.append(acc)
    
    for subj_idx in range(len(train_active_X)):
        print(f"Subject {subj_idx+1} Test Accuracy: {accuracies[subj_idx]:.2f}")

    print(f"\nMean Cross-Validation Accuracy: {np.mean(accuracies):.2f} ± {np.std(accuracies):.2f}")
    

        
    

In [10]:
twofour_crosssession(2)

/tmp/ipykernel_244188/561140518.py:27: RuntimeWarning: Highpass cutoff frequency 100.0 is greater than lowpass cutoff frequency 0.5, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_gdf(filename, preload=True, verbose=False)


['EEG:C3', 'EEG:Cz', 'EEG:C4', 'EOG:ch01', 'EOG:ch02', 'EOG:ch03']
Subject B01: Epoch data shape (120, 3, 751)


/tmp/ipykernel_244188/561140518.py:27: RuntimeWarning: Highpass cutoff frequency 100.0 is greater than lowpass cutoff frequency 0.5, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_gdf(filename, preload=True, verbose=False)


['EEG:C3', 'EEG:Cz', 'EEG:C4', 'EOG:ch01', 'EOG:ch02', 'EOG:ch03']
Subject B02: Epoch data shape (120, 3, 751)


/tmp/ipykernel_244188/561140518.py:27: RuntimeWarning: Highpass cutoff frequency 100.0 is greater than lowpass cutoff frequency 0.5, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_gdf(filename, preload=True, verbose=False)


...

['EEG:C3', 'EEG:Cz', 'EEG:C4', 'EOG:ch01', 'EOG:ch02', 'EOG:ch03']
Subject B07: Epoch data shape (120, 3, 751)


/tmp/ipykernel_244188/561140518.py:119: RuntimeWarning: Highpass cutoff frequency 100.0 is greater than lowpass cutoff frequency 0.5, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_gdf(filename, preload=True, verbose=False)


['EEG:C3', 'EEG:Cz', 'EEG:C4', 'EOG:ch01', 'EOG:ch02', 'EOG:ch03']
Subject B08: Epoch data shape (120, 3, 751)


/tmp/ipykernel_244188/561140518.py:119: RuntimeWarning: Highpass cutoff frequency 100.0 is greater than lowpass cutoff frequency 0.5, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_gdf(filename, preload=True, verbose=False)


['EEG:C3', 'EEG:Cz', 'EEG:C4', 'EOG:ch01', 'EOG:ch02', 'EOG:ch03']
Subject B09: Epoch data shape (120, 3, 751)
Loaded data for 9 subjects.
Subject 1 Test Accuracy: 0.78
Subject 2 Test Accuracy: 0.66
Subject 3 Test Accuracy: 0.58
Subject 4 Test Accuracy: 0.81
Subject 5 Test Accuracy: 0.63
Subject 6 Test Accuracy: 0.75
Subject 7 Test Accuracy: 0.62
Subject 8 Test Accuracy: 0.57
Subject 9 Test Accuracy: 0.65

Mean Cross-Validation Accuracy: 0.67 ± 0.08
